In [5]:
from pyspark.sql.functions import col, min, max, round, lag, when, lit, month, year, concat, lpad
from pyspark.sql.window import Window

SILVER_TABLE = "silver_ae_monthly"
DIM_TRUST = "gold_dim_trust"
FACT_TABLE = "gold_fct_ae_monthly"

print("Config set")

StatementMeta(, dd59c868-d0fb-4ddf-ad2a-20e53fec6664, 12, Finished, Available, Finished, False)

Config set


In [6]:
df_silver = spark.read.table(SILVER_TABLE)

print(f"Rows read from Silver: {df_silver.count()}")
df_silver.printSchema()

StatementMeta(, dd59c868-d0fb-4ddf-ad2a-20e53fec6664, 13, Finished, Available, Finished, False)

Rows read from Silver: 7369
root
 |-- period: string (nullable = true)
 |-- org_code: string (nullable = true)
 |-- parent_org: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- ae_attendances_type1: long (nullable = true)
 |-- ae_attendances_type2: long (nullable = true)
 |-- ae_attendances_other: long (nullable = true)
 |-- ae_attendances_booked_type1: long (nullable = true)
 |-- ae_attendances_booked_type2: long (nullable = true)
 |-- ae_attendances_booked_other: long (nullable = true)
 |-- attendance_over_4hrs_type1: long (nullable = true)
 |-- attendance_over_4hrs_type2: long (nullable = true)
 |-- attendance_over_4hrs_other: long (nullable = true)
 |-- attendance_over_4hrs_booked_type1: long (nullable = true)
 |-- attendance_over_4hrs_booked_type2: long (nullable = true)
 |-- attendance_over_4hrs_booked_other: long (nullable = true)
 |-- patient_waited_4_12hrs_dta: long (nullable = true)
 |-- patient_waited_12plus_hrs_dta: long (nullable = true)
 |-- emergency

In [8]:
df_dim_trust = (
    df_silver
    .select(
        col("org_code"),
        col("parent_org"),
        col("org_name")
    )
    .distinct()
    .orderBy("org_code")
)

print(f"Distinct trusts: {df_dim_trust.count()}")
df_dim_trust.show(20, truncate=False)

StatementMeta(, dd59c868-d0fb-4ddf-ad2a-20e53fec6664, 18, Finished, Available, Finished, False)

Distinct trusts: 229
+--------+------------------------------------+-------------------------------+
|org_code|parent_org                          |org_name                       |
+--------+------------------------------------+-------------------------------+
|8J094   |NHS ENGLAND MIDLANDS                |BADGER LTD                     |
|AAH     |NHS ENGLAND SOUTH WEST              |TETBURY HOSPITAL TRUST LTD     |
|AC008   |NHS ENGLAND NORTH WEST              |ROSSENDALE MINOR INJURIES UNIT |
|ACH01   |NHS ENGLAND SOUTH EAST              |WHITSTABLE MEDICAL PRACTICE    |
|AD913   |NHS ENGLAND LONDON                  |BECKENHAM BEACON UCC           |
|AJN     |NHS ENGLAND NORTH EAST AND YORKSHIRE|WORKINGTON HEALTH LIMITED      |
|AQN04   |NHS ENGLAND SOUTH EAST              |PHL LYMINGTON UTC              |
|ARN03   |NHS ENGLAND NORTH EAST AND YORKSHIRE|LOCAL CARE DIRECT              |
|AXG     |NHS ENGLAND SOUTH WEST              |WILTSHIRE HEALTH & CARE        |
|C82009  |NHS ENGLA

In [32]:
# Window for month-on-month calculations partitioned by trust
window_trust = Window.partitionBy("org_code").orderBy("period_date")

df_fact = (
    df_silver
    .select(
        col("org_code"),
        col("period_date"),
        col("period"),

        # total attendances
        (
            coalesce(col("ae_attendances_type1"), lit(0)) +
            coalesce(col("ae_attendances_type2"), lit(0)) +
            coalesce(col("ae_attendances_other"), lit(0))
        ).alias("total_attendances"),

        # total over 4hrs
        (
            coalesce(col("attendance_over_4hrs_type1"), lit(0)) +
            coalesce(col("attendance_over_4hrs_type2"), lit(0)) +
            coalesce(col("attendance_over_4hrs_other"), lit(0))
        ).alias("total_over_4hrs"),

        # total booked attendances
        (
            coalesce(col("ae_attendances_booked_type1"), lit(0)) +
            coalesce(col("ae_attendances_booked_type2"), lit(0)) +
            coalesce(col("ae_attendances_booked_other"), lit(0))
        ).alias("total_booked_attendances"),

        # total booked over 4hrs
        (
            coalesce(col("attendance_over_4hrs_booked_type1"), lit(0)) +
            coalesce(col("attendance_over_4hrs_booked_type2"), lit(0)) +
            coalesce(col("attendance_over_4hrs_booked_other"), lit(0))
        ).alias("total_booked_over_4hrs"),

        # raw KPI columns
        col("ae_attendances_type1"),
        col("ae_attendances_type2"),
        col("ae_attendances_other"),
        col("ae_attendances_booked_type1"),
        col("ae_attendances_booked_type2"),
        col("ae_attendances_booked_other"),
        col("attendance_over_4hrs_type1"),
        col("attendance_over_4hrs_type2"),
        col("attendance_over_4hrs_other"),
        col("attendance_over_4hrs_booked_type1"),
        col("attendance_over_4hrs_booked_type2"),
        col("attendance_over_4hrs_booked_other"),
        col("patient_waited_4_12hrs_dta"),
        col("patient_waited_12plus_hrs_dta"),
        col("emergency_admissions_type1"),
        col("emergency_admissions_type2"),
        col("emergency_admissions_other"),
        col("other_emergency_admissions"),
    )
    # 4-hour target % — attendances seen within 4hrs / total
    .withColumn("four_hr_target_pct",
        round(
            when(col("total_attendances") > 0,
                (col("total_attendances") - col("total_over_4hrs")) * 100.0 / col("total_attendances")
            ).otherwise(lit(None)),
        2)
    )
    # 12-hour breach rate
    .withColumn("twelve_hr_breach_rate",
        round(
            when(col("total_attendances") > 0,
                col("patient_waited_12plus_hrs_dta") * 100.0 / col("total_attendances")
            ).otherwise(lit(None)),
        2)
    )
    # month-on-month change in total attendances
    .withColumn("prev_month_attendances",
        lag("total_attendances", 1).over(window_trust)
    )
    .withColumn("mom_attendance_change_pct",
        round(
            when(col("prev_month_attendances") > 0,
                (col("total_attendances") - col("prev_month_attendances")) * 100.0 / col("prev_month_attendances")
            ).otherwise(lit(None)),
        2)
    )
    # month-on-month change in 4hr target performance
    .withColumn("prev_month_4hr_pct",
        lag("four_hr_target_pct", 1).over(window_trust)
    )
    .withColumn("mom_4hr_target_change",
        round(col("four_hr_target_pct") - col("prev_month_4hr_pct"), 2)
    )
    .drop("prev_month_attendances", "prev_month_4hr_pct")
)

print(f"Fact table rows before filter: {df_fact.count()}")

# Filter out rows with no attendances — these are aggregate/summary org codes
df_fact_clean = df_fact.filter(col("total_attendances") > 0)

print(f"Fact table rows after filter: {df_fact_clean.count()}")

df_fact_clean.select(
    "org_code", "period_date", "total_attendances",
    "four_hr_target_pct", "twelve_hr_breach_rate",
    "mom_attendance_change_pct", "mom_4hr_target_change"
).orderBy(
    col("period_date").desc(),
    col("org_code")
).show(20, truncate=False)

StatementMeta(, dd59c868-d0fb-4ddf-ad2a-20e53fec6664, 42, Finished, Available, Finished, False)

Fact table rows before filter: 7369
Fact table rows after filter: 6890
+--------+-----------+-----------------+------------------+---------------------+-------------------------+---------------------+
|org_code|period_date|total_attendances|four_hr_target_pct|twelve_hr_breach_rate|mom_attendance_change_pct|mom_4hr_target_change|
+--------+-----------+-----------------+------------------+---------------------+-------------------------+---------------------+
|AAH     |2024-03-01 |532              |99.81             |0.0                  |16.92                    |0.25                 |
|ACH01   |2024-03-01 |4746             |100.0             |0.0                  |13.24                    |0.0                  |
|AD913   |2024-03-01 |4049             |99.09             |0.0                  |13.39                    |-0.41                |
|AJN     |2024-03-01 |334              |100.0             |0.0                  |9.15                     |0.0                  |
|AQN04   |2024-03-0

In [34]:
# Write dim_trust
df_dim_trust.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_TRUST)

# Write fct_ae_monthly
(
    df_fact_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_TABLE)
)

print(f"✅ dim_trust written: {spark.read.table(DIM_TRUST).count()} rows")
print(f"✅ fct_ae_monthly written: {spark.read.table(FACT_TABLE).count()} rows")

StatementMeta(, dd59c868-d0fb-4ddf-ad2a-20e53fec6664, 44, Finished, Available, Finished, False)

✅ dim_trust written: 229 rows
✅ fct_ae_monthly written: 6890 rows
